# MNIST Digit Recognition — Exploratory Notebook

This notebook is kept for exploratory analysis and walkthrough purposes. The actual, tested pipeline logic lives in [`src/mnist_digit_recognition/`](../src/mnist_digit_recognition/) — this notebook now **imports and calls that package** instead of duplicating its logic, so there is a single source of truth.

For a fast, reproducible run outside Jupyter, use:
```bash
python scripts/train.py --model all
python scripts/serve.py --model random_forest
```
See the top-level [README](../README.md) for full setup instructions.


In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable when running this notebook directly
# from the notebooks/ directory (no editable install required).
SRC_PATH = Path.cwd().parent / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


# **1. Data Preparation**

## 1.1. Importing Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mnist_digit_recognition.config import load_config
from mnist_digit_recognition.data import (
    fit_scaler,
    load_mnist,
    scale_features,
    split_dataset,
)
from mnist_digit_recognition.evaluate import (
    evaluate_model,
    most_common_misclassifications,
    plot_confusion_matrix,
)
from mnist_digit_recognition.logging_config import setup_logging
from mnist_digit_recognition.models import (
    ModelBundle,
    ModelName,
    build_model,
    needs_scaling,
    save_bundle,
    train_model,
)

setup_logging("INFO")
config = load_config()
config.ensure_directories()


## 1.2. Loading Dataset

In [ ]:
X, y = load_mnist(config.data)


## 1.3. Splitting Dataset

In [ ]:
dataset = split_dataset(X, y, config.data)


## 1.4. Scaling Data

In [ ]:
scaler = fit_scaler(dataset.X_train)
X_train_scaled, X_test_scaled = scale_features(scaler, dataset.X_train, dataset.X_test)


## 1.5. Summary

In [ ]:
print("Training set shape:", dataset.X_train.shape)
print("Test set shape:", dataset.X_test.shape)
print("Unique labels:", np.unique(dataset.y_train))


#  **2. Model Training**

## 2.1. Train SGDClassifier

In [ ]:
sgd_model = build_model(ModelName.SGD, config.random_forest, config.sgd)
train_model(sgd_model, X_train_scaled, dataset.y_train)
sgd_pred = sgd_model.predict(X_test_scaled)
sgd_result = evaluate_model("SGDClassifier", dataset.y_test, sgd_pred)
print(f"SGDClassifier Accuracy: {sgd_result.accuracy:.4f}")


## 2.2. Train RandomForestClassifier

In [ ]:
rf_model = build_model(ModelName.RANDOM_FOREST, config.random_forest, config.sgd)
train_model(rf_model, dataset.X_train, dataset.y_train)
rf_pred = rf_model.predict(dataset.X_test)
rf_result = evaluate_model("RandomForestClassifier", dataset.y_test, rf_pred)
print(f"RandomForestClassifier Accuracy: {rf_result.accuracy:.4f}")


# **3. Model Evaluation (Confusion Matrix, Classification Report, Metrics)**

## 3.1. Evaluate SGDClassifier (Linear SVM)

In [ ]:
print(sgd_result.classification_report)
plot_confusion_matrix(sgd_result, config.reports_dir / "sgd_confusion_matrix.png")
plt.show()


##  3.2. Evaluate RandomForestClassifier

In [ ]:
print(rf_result.classification_report)
plot_confusion_matrix(rf_result, config.reports_dir / "random_forest_confusion_matrix.png")
plt.show()


# **4. Error Visualization (Worst-Case Misclassifications)**

## 4.1: Identify Misclassified Instances (RandomForestClassifier)

In [ ]:
most_common_errors = most_common_misclassifications(dataset.y_test, rf_pred)
most_common_errors


## 4.2: Plot Example Misclassified Images (e.g., 9→4)

In [ ]:
target_actual, target_predicted = most_common_errors[0][0]

misclassified_idx = np.where(rf_pred != dataset.y_test)[0]
target_indices = [
    idx
    for idx in misclassified_idx
    if dataset.y_test[idx] == target_actual and rf_pred[idx] == target_predicted
]

plt.figure(figsize=(10, 4))
for i, idx in enumerate(target_indices[:5]):
    image = dataset.X_test[idx].reshape(28, 28)
    plt.subplot(1, 5, i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"Actual: {target_actual}, Pred: {target_predicted}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
figures_dir = config.reports_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
for i in range(5):
    img = dataset.X_test[i].reshape(28, 28)
    label = dataset.y_test[i]
    filename = figures_dir / f"mnist_test_digit_{i}_label_{label}.png"
    plt.imsave(filename, img, cmap="gray")
    print(f"Saved {filename}")


## 5.1. Load Trained Model + Scaler

In [ ]:
sgd_bundle = ModelBundle(
    name=ModelName.SGD, model=sgd_model, needs_scaling=needs_scaling(ModelName.SGD), scaler=scaler
)
rf_bundle = ModelBundle(
    name=ModelName.RANDOM_FOREST, model=rf_model, needs_scaling=needs_scaling(ModelName.RANDOM_FOREST), scaler=None
)
save_bundle(sgd_bundle, config.model_dir)
save_bundle(rf_bundle, config.model_dir)


## 5.2. Define Preprocessing + Prediction Function + Gradio Interface

In [ ]:
from mnist_digit_recognition.app import build_interface

iface = build_interface(rf_bundle)
# iface.launch()  # uncomment to launch interactively; see scripts/serve.py for a CLI entrypoint
